In [1]:
# ============================================================
# CONTEXT AWARE COLORIZATION USING USER INPUT + GUI
# ============================================================

import sys
import cv2
import torch
import numpy as np

from PIL import Image

from transformers import (
    SegformerImageProcessor,
    SegformerForSemanticSegmentation
)

from PyQt5.QtWidgets import (
    QApplication,
    QWidget,
    QLabel,
    QPushButton,
    QFileDialog,
    QVBoxLayout,
    QHBoxLayout
)

from PyQt5.QtGui import QPixmap, QImage
from PyQt5.QtCore import Qt

# -------------------------------------------------
# Load SegFormer
# -------------------------------------------------

print("Loading SegFormer...")

processor = SegformerImageProcessor.from_pretrained(
    "nvidia/segformer-b2-finetuned-ade-512-512"
)

model = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/segformer-b2-finetuned-ade-512-512"
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model.to(device)
model.eval()

print("Model loaded successfully")

# -------------------------------------------------
# Color dictionary
# -------------------------------------------------

COLORS = {

    # sky
    2: [255,220,150],

    # tree
    4: [0,180,0],

    # grass
    9: [50,220,50],

    # road
    6: [120,120,120],

    # building
    1: [180,160,140],

    # water
    21: [255,100,0],

    # mountain
    16: [120,90,70]
}


# -------------------------------------------------
# Segmentation
# -------------------------------------------------

def segment_image(image):

    rgb = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2RGB
    )

    pil = Image.fromarray(rgb)

    inputs = processor(
        images=pil,
        return_tensors="pt"
    )

    inputs = {
        k:v.to(device)
        for k,v in inputs.items()
    }

    with torch.no_grad():

        outputs = model(**inputs)

    logits = outputs.logits

    logits = torch.nn.functional.interpolate(
        logits,
        size=image.shape[:2],
        mode="bilinear",
        align_corners=False
    )

    prediction = logits.argmax(
        dim=1
    )[0]

    return prediction.cpu().numpy()


# -------------------------------------------------
# Context-aware colorization
# -------------------------------------------------

def context_colorize(image):

    mask = segment_image(image)

    output = image.copy()

    for class_id, color in COLORS.items():

        output[mask == class_id] = color

    # blending
    output = cv2.addWeighted(
        image,
        0.4,
        output,
        0.6,
        0
    )

    return output


# -------------------------------------------------
# GUI
# -------------------------------------------------

class ColorizationGUI(QWidget):

    def __init__(self):

        super().__init__()

        self.setWindowTitle(
            "Context-Aware Scene Colorization"
        )

        self.setGeometry(
            100,
            50,
            1400,
            850
        )

        self.image = None
        self.output_image = None

        # ----------------------------------
        # Dark Theme
        # ----------------------------------

        self.setStyleSheet("""

            QWidget{
                background-color:#1E1E2F;
                color:white;
                font-size:15px;
            }

            QPushButton{
                background-color:#0078D7;
                color:white;
                border-radius:15px;
                padding:12px;
                font-size:15px;
                font-weight:bold;
            }

            QPushButton:hover{
                background-color:#00A2FF;
            }

            QLabel{
                color:white;
            }

        """)

        # ----------------------------------
        # Title
        # ----------------------------------

        self.title = QLabel(
            "Context-Aware Complex Scene Colorization using SegFormer"
        )

        self.title.setAlignment(
            Qt.AlignCenter
        )

        self.title.setStyleSheet("""
            font-size:28px;
            font-weight:bold;
            color:#00D9FF;
            padding:15px;
        """)

        # ----------------------------------
        # Original Image
        # ----------------------------------

        self.original_label = QLabel(
            "Original Image"
        )

        self.original_label.setAlignment(
            Qt.AlignCenter
        )

        self.original_label.setFixedSize(
            550,
            500
        )

        self.original_label.setStyleSheet("""
            background:white;
            color:black;
            border:3px solid #00D9FF;
            border-radius:20px;
        """)

        # ----------------------------------
        # Output Image
        # ----------------------------------

        self.result_label = QLabel(
            "Colorized Output"
        )

        self.result_label.setAlignment(
            Qt.AlignCenter
        )

        self.result_label.setFixedSize(
            550,
            500
        )

        self.result_label.setStyleSheet("""
            background:white;
            color:black;
            border:3px solid #00FF80;
            border-radius:20px;
        """)

        # ----------------------------------
        # Buttons
        # ----------------------------------

        self.upload_btn = QPushButton(
            "📁 Upload Image"
        )

        self.save_btn = QPushButton(
            "💾 Save Output"
        )

        self.upload_btn.setStyleSheet("""
            QPushButton{
                background:#3A86FF;
                color:white;
                border-radius:15px;
                padding:12px;
                font-size:15px;
                font-weight:bold;
            }

            QPushButton:hover{
                background:#5AA3FF;
            }
        """)

        self.save_btn.setStyleSheet("""
            QPushButton{
                background:#FF006E;
                color:white;
                border-radius:15px;
                padding:12px;
                font-size:15px;
                font-weight:bold;
            }

            QPushButton:hover{
                background:#FF3D93;
            }
        """)

        # ----------------------------------
        # Status Label
        # ----------------------------------

        self.status_label = QLabel(
            "Ready"
        )

        self.status_label.setAlignment(
            Qt.AlignCenter
        )

        self.status_label.setStyleSheet("""
            color:#FFD166;
            font-size:16px;
            padding:10px;
        """)

        # ----------------------------------
        # Layouts
        # ----------------------------------

        image_layout = QHBoxLayout()

        image_layout.addWidget(
            self.original_label
        )

        image_layout.addWidget(
            self.result_label
        )

        button_layout = QHBoxLayout()

        button_layout.addWidget(
            self.upload_btn
        )

        button_layout.addWidget(
            self.save_btn
        )

        main_layout = QVBoxLayout()

        main_layout.addWidget(
            self.title
        )

        main_layout.addLayout(
            image_layout
        )

        main_layout.addSpacing(
            20
        )

        main_layout.addLayout(
            button_layout
        )

        main_layout.addWidget(
            self.status_label
        )

        self.setLayout(
            main_layout
        )

        # ----------------------------------
        # Connections
        # ----------------------------------

        self.upload_btn.clicked.connect(
            self.load_image
        )

        self.save_btn.clicked.connect(
            self.save_output
        )

    def load_image(self):
        file_path, _ = QFileDialog.getOpenFileName(
            self,
            "Open Image",
            "",
            "Images (*.jpg *.png *.jpeg)"
        )

        if file_path:

            self.image = cv2.imread(
                file_path
            ) 

            self.display_image(
                self.image,
                self.original_label
            )

            self.output_image = context_colorize(
                self.image
            )

            self.display_image(
                self.output_image,
                self.result_label
            )

            self.status_label.setText(
                "Colorization Completed"
            )
            
    def display_image(
        self,
        image,
        label
        ):

        rgb = cv2.cvtColor(
            image,
            cv2.COLOR_BGR2RGB
        )   

        h, w, ch = rgb.shape

        bytes_per_line = ch * w

        qimage = QImage(
             rgb.data,
             w,
             h,
             bytes_per_line,
             QImage.Format_RGB888
        )

        pixmap = QPixmap.fromImage(
             qimage
        )

        label.setPixmap(
             pixmap.scaled(
                  label.width(),
                  label.height(),
                  Qt.KeepAspectRatio,
                  Qt.SmoothTransformation
             )
           )

    def save_output(self):

        if self.output_image is None:
            return

        save_path, _ = QFileDialog.getSaveFileName(
             self,
             "Save Image",
             "",
             "PNG Files (*.png);;JPEG Files (*.jpg)"
        )

        if save_path:
            cv2.imwrite(
               save_path,
               self.output_image
            )

            self.status_label.setText(
                "Image Saved Successfully"
            )


# -------------------------------------------------
# Main
# -------------------------------------------------

app = QApplication(sys.argv)

window = ColorizationGUI()

window.show()

sys.exit(app.exec())

Loading SegFormer...


Loading weights:   0%|          | 0/380 [00:00<?, ?it/s]

Model loaded successfully


SystemExit: 0

C:\Users\SERVER\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\IPython\core\interactiveshell.py:3709: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
